In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt



In [ ]:
def generate_mixture_dataset(n_samples=100,projection= lambda x: x,proj_sd=0.):

    ## play around with parameters here
    generator = np.random.default_rng()
    
    mixture_weights = [0.25,0.1,0.3,0.35]
    mu1,mu2,mu3,mu4 = np.array([-0.05,1]),np.array([1,0]),np.array([2,0]),np.array([-0.5,-0.25])
    cov1,cov2,cov3,cov4 = np.array([[1,0],[0,1]]),np.array([[1,0.35],[0.35,1]]),np.array([[1,0.25],[0.25,1]]),np.array([[1,-0.75],[-0.75,1]])

    mus = np.stack([mu1,mu2,mu3,mu4],axis=0)
    covs = np.stack([cov1,cov2,cov3,cov4],axis=0)/5

    sample_labels = generator.choice(4,n_samples,replace=True,p=mixture_weights)
    #print(samples.shape)
    print(np.sum(sample_labels == 0)/n_samples)
    print(np.sum(sample_labels == 1)/n_samples)
    print(np.sum(sample_labels == 2)/n_samples)
    print(np.sum(sample_labels == 3)/n_samples)

    mu_samples,cov_samples = mus[sample_labels],covs[sample_labels]

    data_samples = generator.multivariate_normal(mean=np.zeros((2,)),cov=np.eye(2),size=(n_samples))
    #print(data_samples.shape)
    data_samples = mu_samples + np.einsum('nkp,np->nk',cov_samples,data_samples)

    projected_samples = projection(data_samples,proj_sd)
    #print(data_samples.shape)
    return data_samples,projected_samples,sample_labels

In [ ]:
class LinearProjection():

    def __init__(self,data_dim,project_dim):

        self.data_dim = data_dim
        self.project_dim=project_dim
        self.w = np.random.randn(data_dim,project_dim)
        #self.noise_sd = noise_sd

    def __call__(self,data,noise_sd=0.):

        return self.project(data,noise_sd)

    def project(self,data,noise_sd=0.):

        # expects data to be N x d, projection weight to be d x P

        return data @ self.w + noise_sd * np.random.randn(data.shape[0],self.project_dim)

    def un_project(self,data):

        return data @ self.w.T @ np.linalg.pinv(self.w @ self.w.T)

In [ ]:
project_dim=500
lp = LinearProjection(data_dim=2,project_dim=project_dim)
data,projected,labels = generate_mixture_dataset(n_samples=10000,projection=lp,proj_sd=1.5)
unprojected = lp.un_project(projected)
w = lp.w.T

In [ ]:
l1_norm = np.amax(np.sum(np.abs(w),axis=0))
linf_norm = np.amax(np.sum(np.abs(w),axis=1))
print(l1_norm,linf_norm)

In [ ]:
%matplotlib inline

ax = plt.gca()
for ii in range(4):
    inds = labels == ii
    ax.scatter(data[inds,0],data[inds,1],label=f'cluster {ii+1}')

ax.legend()
plt.show()
plt.close()

ax = plt.gca()
for ii in range(4):
    inds = labels == ii
    ax.scatter(unprojected[inds,0],unprojected[inds,1],label=f'cluster {ii+1}')

ax.legend()
plt.show()
plt.close()


In [ ]:
from sklearn.mixture import GaussianMixture as GMM

model1 = GMM(n_components=4,covariance_type='full',n_init=10)
model1.fit(projected)
pred_labels=model1.predict(projected)
print(f"mixture weights: {model1.weights_}")


In [ ]:
ax = plt.gca()
for ii in range(4):
    inds = pred_labels == ii
    ax.scatter(unprojected[inds,0],unprojected[inds,1],label=f'cluster {ii+1}')

ax.legend()
plt.show()
plt.close()

In [ ]:
from models.vae import *
from data.data_utils import *

loaders = get_loaders(projected,num_workers=3)


In [ ]:
print(labels.shape)

In [ ]:
def power_iter(weight,n_power_iters=50):

        x = torch.randn((weight.shape[1]),device=weight.device)
        print(x.shape)
        print(x)
        for ii in range(n_power_iters):
            x = weight.T @ weight @ x
            print(x)
        print(x.shape)
        print(x)
        print(torch.linalg.norm(weight @ x))
        return torch.linalg.norm(weight @ x)/torch.linalg.norm(x)

In [ ]:
max_sv_proj = power_iter(torch.from_numpy(w.T).to(torch.float32),n_power_iters=5)

In [ ]:
print(w.shape)

In [ ]:
print(max_sv_proj)

In [ ]:
dec1 = LipschitzDecoder(n_layers=7,data_dim=10,latent_dim=2,hidden_dim=5,max_lipschitz=5)

#pre,post,names = dec1.lipschitz_constrain()

In [ ]:
for pr,po,n in zip(pre,post,names):

    fig, (ax1,ax2) = plt.subplots(nrows=1,ncols=2)
    vmin,vmax = np.amin(pr),np.amax(pr)
    ax1.matshow(pr,vmin=vmin,vmax=vmax)
    ax2.matshow(po,vmin=vmin,vmax=vmax)
    #ax3.matshow(pr - po)
    fig.suptitle(f"layer {n}")

    plt.show()
    plt.close()

In [ ]:
l2_norm_fnc = lambda w: power_iter(w,n_power_iters=50)
l1_norm_fnc = lambda w: torch.amax(torch.sum(w.abs(),dim=0))
linf_norm_fnc = lambda w: torch.amax(torch.sum(w.abs(),dim=1))
dec1 = LipschitzDecoder(n_layers=7,data_dim=10,latent_dim=2,hidden_dim=5,max_lipschitz=l1_norm,norm_func=l1_norm_fnc)


In [ ]:
dec1.regularize()

In [ ]:
#from train.losses import ELBO
#from train.train import train
## add in gradient clipping
import torch.nn as nn
from train.losses import ELBO
from train.train import train
from eval.eval import train_test_plot,embedding_plot

from torch.distributions.lowrank_multivariate_normal import LowRankMultivariateNormal

precisions = [1e-1,5e-1,1,5,1e1,5e1] #,1e2,1e3]
lrs = [1e-3]*5 + [1e-4] #,1e-3,1e-3,1e-4,1e-5]

np.amax(np.sum(np.abs(w),axis=0))
l2_norm_fnc = lambda w: power_iter(w,n_power_iters=50)
l1_norm_fnc = lambda w: torch.amax(torch.sum(w.abs(),dim=0))
linf_norm_fnc = lambda w: torch.amax(torch.sum(w.abs(),dim=1))
lipschitz_coef = 10*l1_norm

for precision,lr in zip(precisions,lrs):

    loss = lambda target,model_out: ELBO(target,model_out,recon_precision=precision)
    


    okay_model=False
    max_tries=10
    curr_try = 0
    while not okay_model & (curr_try < max_tries):
        try:
            enc3 = ProbabilisticEncoder(n_layers_shared=4,n_layers_private=3,data_dim=500,hidden_dim=125,latent_dim=2)
            dec3 = LipschitzDecoder(n_layers=7,data_dim=500,latent_dim=2,hidden_dim=25,activation=nn.GELU(),max_lipschitz=lipschitz_coef,norm_func=l1_norm_fnc)
            vae3 = VariationalAutoEncoder(enc3,dec3,LowRankMultivariateNormal)
            vae3,opt3,scheduler3,recons3,regs3 = train(vae3,loaders,loss,nEpochs=150,val_freq=10,lr=lr,max_norm_grad=1e-4)
            okay_model=True
        except:
            curr_try +=1
            print("bad params,retraining")
            
    train_test_plot(recons3,regs3,label=f"{lipschitz_coef}-lipschitz linear decoder, recon precision = {precision}")
    
    valid_grads = True

    ### maybe add some checking here to make sure all the gradients are good
    okay_model=False
    curr_try = 0
    while not okay_model& (curr_try < max_tries):
        try:
            enc1 = ProbabilisticEncoder(n_layers_shared=4,n_layers_private=3,data_dim=500,hidden_dim=125,latent_dim=2)
            dec1 = Decoder(n_layers=7,data_dim=500,latent_dim=2,hidden_dim=125)
            vae1 = VariationalAutoEncoder(enc1,dec1,LowRankMultivariateNormal)
            vae1,opt,scheduler,recons,regs = train(vae1,loaders,loss,nEpochs=150,val_freq=10,lr=1e-3,max_norm_grad=1e-2)
            okay_model=True
        except:
            curr_try += 1
            print("bad params,retraining")
        


    train_test_plot(recons,regs,label=f"arbitrary decoder, recon precision = {precision}")

    
    
    okay_model=False
    curr_try = 0
    while not okay_model & (curr_try < max_tries):
        try:
            enc2 = ProbabilisticEncoder(n_layers_shared=4,n_layers_private=3,data_dim=500,hidden_dim=125,latent_dim=2)
            dec2 = Decoder(n_layers=7,data_dim=500,latent_dim=2,hidden_dim=125,activation=nn.Identity())
            vae2 = VariationalAutoEncoder(enc2,dec2,LowRankMultivariateNormal)
            vae2,opt2,scheduler2,recons2,regs2 = train(vae2,loaders,loss,nEpochs=150,val_freq=10,lr=lr,max_norm_grad=1e-2)
            okay_model=True
        except:
            curr_try += 1
            print("bad params,retraining")

    train_test_plot(recons2,regs2,label=f"deep linear decoder, recon precision = {precision}")
    
    

    embeddings1 = vae1.encoder(torch.from_numpy(projected).to(vae1.device).to(torch.float32))[0]
    reprojections1 = vae1.decoder(embeddings1).detach().cpu().numpy() ## change these to pc plots
    embeddings1 = embeddings1.detach().cpu().numpy()
    embedding_plot(embeddings1,reprojections1,data,labels,label=f'arbitrary decoder, recon precision = {precision}')

    embeddings2 = vae2.encoder(torch.from_numpy(projected).to(vae2.device).to(torch.float32))[0]
    reprojections2 = vae2.decoder(embeddings2).detach().cpu().numpy()
    embeddings2 = embeddings2.detach().cpu().numpy()

    embedding_plot(embeddings2,reprojections2,data,labels,label=f'deep linear decoder, recon precision = {precision}')

    embeddings3 = vae3.encoder(torch.from_numpy(projected).to(vae3.device).to(torch.float32))[0]
    reprojections3 = vae3.decoder(embeddings3).detach().cpu().numpy()
    embeddings3 = embeddings3.detach().cpu().numpy()

    embedding_plot(embeddings3,reprojections3,data,labels,label=f'{lipschitz_coef}-lipschitz,nonlinear decoder, recon precision = {precision}')


In [ ]:
def save_model(model,optimizer,location):


    sd = {'ac': model.state_dict(),
          'opt':optimizer.state_dict(),
          'encoder specs': model.encoder.spec_dict,
          'decoder specs': model.decoder.spec_dict,
          'ac specs': model.spec_dict
    }

    torch.save(sd,location)

def load_model(location):

    sd = torch.load(location,weights_only=False)
    model_params = sd['ac']
    opt_params = sd['opt']
    encoder_details = sd['encoder specs']
    decoder_details = sd['decoder specs']
    ac_details = sd['ac specs']
    encoder_type = encoder_details['type']
    decoder_type = decoder_details['type']
    ac_type = ac_details['type']

    if encoder_type == 'MLP':
        enc = Encoder(n_layers=encoder_details['n_layers'],
                      data_dim=encoder_details['data_dim'],
                      hidden_dim=encoder_details['hidden_dim'],
                      latent_dim=encoder_details['latent_dim'],
                      activation=encoder_details['activation'],
                      device=encoder_details['device'])

    elif encoder_type == 'probabilistic':
        enc = ProbabilisticEncoder(n_layers_shared=encoder_details['n_layers_shared'],
                             n_layers_private=encoder_details['n_layers_private'],
                      data_dim=encoder_details['data_dim'],
                      hidden_dim=encoder_details['hidden_dim'],
                      latent_dim=encoder_details['latent_dim'],
                      activation=encoder_details['activation'],
                      device=encoder_details['device'])
    else:
        raise NotImplementedError
    
    if decoder_type == 'MLP':
        dec = Decoder(n_layers=decoder_details['n_layers'],
                      data_dim=decoder_details['data_dim'],
                      hidden_dim=decoder_details['hidden_dim'],
                      latent_dim=decoder_details['latent_dim'],
                      activation=decoder_details['activation'],
                      device=decoder_details['device'])
    else:
        raise NotImplementedError
    
    if ac_type == 'autoencoder':
        model = AutoEncoder(enc,dec,device=ac_details['device'])

    elif ac_type =='VAE':
        model = VariationalAutoEncoder(enc,dec,device=ac_details['device'],latent_distribution=ac_details['latent_dist'])

    model.load_state_dict(model_params)
    opt=Adam(model.parameters(),lr=1e-3)
    opt.load_state_dict(opt_params)
    scheduler = ReduceLROnPlateau(opt,factor=0.75,patience=5,min_lr=1e-10)


    return model,opt,scheduler

In [ ]:
vae_loaded,opt_loaded,scheduler_loaded = load_model('/home/miles/Downloads/vae_test1.tar')

In [ ]:
save_model(vae,opt,'/home/miles/Downloads/vae_test1.tar')

In [ ]:
embeddings = vae_loaded.encoder(torch.from_numpy(projected).to(vae.device).to(torch.float32))[0]
embeddings= embeddings.detach().cpu().numpy()
print(embeddings.shape,data.shape)
ax = plt.gca()
for ii in range(4):
    inds = labels == ii
    ax.scatter(embeddings[inds,0],embeddings[inds,1],label=f'cluster {ii+1}')

ax.legend()
plt.show()
plt.close()